# Trouble-shooting the AI-Synbio LIMS Mirror DB: Data integrity, syncing operations, archiving

Want to be able to 
- Create db from Google Sheets.
- Sync sheets with db
    - schema changes are implemented
    - updated rows are updated, not inserted
    - new rows are inserted
    - deleted rows are marked as deleted
- Sync daemon syncs at defined intervals
- Make copies of the database at defined intervals (archives)
    - Create a copy every (24 hours)
    - Retain only copies at defined intervals and delete all other intermittent copies using archive daemon

## What I have observed to be broken/missing
- At each sync, all rows are re-inserted into the db, even if they haven't changed (and their row hash hasn't changed) resulting in a gargantuan db. I killed the daemon.
- There are problems with file paths to the sync.log which the config defines as /storage/synbio/sync.log. It writes to that location, but for some reason, somewhere else in the code it expects sync.log to be in folder that contains the sync module.
- Starting and stopping the sync daemon doesn't immediately take place, and errors may occur that don't change the global status of _daemon_running.
- I want to have control over the daemon processes and kill them if necessary by hand. There have been times when I was thinking that there might be multiple sync daemon running simultaneously, all writing to the same database, log file - causing problems.
- Automatic copying of the mirror db and cleanup of the archive at the specified intervals is not implemented.
  
## How to fix things - with AI??

- Merge dev into main. Push to GitHub. Pull to laptop.
- Ask copilot:
  - How does the scheduler underlying the sync daemon work? With the current code version, is it theoretically possible to have multiple sync daemons running simultaneously? How can I ensure that this never happens?
  - During sync, all LIMS rows are re-inserted causing the db to balloon. They shouldn't be if the row hash is the same. But they are nonetheless. Why? How can this be prevented?
  - If starting or stopping the sync daemon with start_sync_daemon or stop_sync_daemon() run into an error, the status of the global _daemon_running variable does not necessarily change. How do I ensure the _daemon_running variable represents the actual daemon state?
  - Create an archive daemon (similar to the sync daemon) that makes copies of the database at a specified location and at specified intervals and keeps only those copies specified in the retention policy.


